# Business Entity Resolution: Kaggle runner

**Setup (once):**
1. Upload `student_resource/dataset/` (the `train/` and `test/` folders) as a **private** Kaggle Dataset, then *Add Input* it to this notebook.
2. Settings → **Internet on** (needed for `git clone` and `pip install`). Accelerator: **None (CPU)**; this pipeline does not use a GPU.
3. Set `DATA_SLUG` below to the dataset folder name under `/kaggle/input/`.

Every stage writes to `/kaggle/working` and is skipped if its output already exists. If a session dies, re-run all and it resumes. To reuse outputs across sessions, save a version and add this notebook's output as an input.

In [ ]:
DATA_SLUG = "amazon-ml-2026-er"   # <-- folder name under /kaggle/input
REPO = "https://github.com/stack-ajit/business_entity_resolution.git"
BRANCH = "main"

In [ ]:
!pip install -q anyascii==0.3.3 sparse-dot-topn==1.2.0 rapidfuzz lightgbm
import os, glob
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
# find the folder that holds train/ and test/ (works whatever the upload nesting is)
cand = [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/{DATA_SLUG}/**/train", recursive=True)]
assert cand, "train/ folder not found - check DATA_SLUG"
os.environ["ER_DATA_DIR"] = cand[0]
os.environ["ER_WORK_DIR"] = "/kaggle/working"
print("data:", os.environ["ER_DATA_DIR"])
!ls $ER_DATA_DIR/train $ER_DATA_DIR/test
!nproc && free -g

## 1. Sample for blocking evaluation (10K train S1)

In [ ]:
%cd /kaggle/working/repo
!test -f /kaggle/working/sample/sample_ground_truth.tsv || python src/data/create_sample.py

## 2. Build train index + measure blocking recall (full 10.3M train S2/S3)

In [ ]:
!python src/blocking/evaluate_blocking.py --top-k 100

## 3. Labelled training pairs for the matching model

In [ ]:
!test -f /kaggle/working/cache/train_pairs.parquet || python src/matching/build_training_set.py --n-s1 150000 --top-k 50